# Module 27 — ReWOO: reasoning without observation

**THE ONE IDEA:** ReWOO enumerates **every** tool call up front, runs the independent ones
**in parallel**, and then does **one** reasoning pass over all the results together.

ReAct does N reasoning calls for N tools. ReWOO does **two**, total — one to plan, one to
solve — no matter how many tools run.

```
ReAct   plan→act→observe→plan→act→observe→plan→answer     N+1 LLM calls, serial
ReWOO   PLAN(1 call) → [E1 ‖ E2 ‖ E3] → SOLVE(1 call)     2 LLM calls, parallel tools
```

The cost of that: it is **more** rigid than module 26. Not only is the plan fixed, the
reasoning never sees an intermediate result in time to change course.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import re, time
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel
from typing import Literal
from _providers import get_client
from _tools import run_tool

client, MODEL, _ = get_client("openai")

class Ev(BaseModel):
    id: str                                   # "E1"
    tool: Literal["search_policy", "calculate"]
    arg: str                                  # may contain #E1 placeholders
class Blueprint(BaseModel):
    evidence: list[Ev]

TASK = ("Compare the year-2 early repayment charge with the year-5 charge on a "
        "250000 loan, and state the maximum LTV for a first-time buyer.")

## Phase 1 — PLAN: enumerate all evidence, once

In [ ]:
s = Blueprint.model_json_schema(); s["additionalProperties"] = False
r = client.chat.completions.create(model=MODEL, max_tokens=700,
    response_format={"type": "json_schema",
                     "json_schema": {"name": "bp", "strict": True, "schema": s}},
    messages=[{"role": "user", "content":
        "List EVERY tool call needed, as E1, E2, ... Use #E1 to reference an earlier "
        "result. Independent items must not reference each other.\n"
        "TOOLS: search_policy(query), calculate(expression)\n\nTASK: " + TASK}])
bp = Blueprint.model_validate_json(r.choices[0].message.content)
for e in bp.evidence:
    dep = re.findall(r"#(E\d+)", e.arg)
    print(f"  {e.id}: {e.tool:14} {e.arg[:34]:34} depends_on={dep or '-'}")

## Phase 2 — EXECUTE: independent evidence runs concurrently

In [ ]:
def deps(e): return set(re.findall(r"#(E\d+)", e.arg))
done, pending, waves = {}, list(bp.evidence), 0
t0 = time.time()
while pending:
    ready = [e for e in pending if deps(e) <= done.keys()]
    if not ready: break
    waves += 1
    def run(e):
        arg = e.arg
        for k, v in done.items(): arg = arg.replace(f"#{k}", str(v))
        key = "query" if e.tool == "search_policy" else "expression"
        return e.id, run_tool(e.tool, {key: arg})
    with ThreadPoolExecutor(max_workers=len(ready)) as pool:
        for eid, out in pool.map(run, ready): done[eid] = out
    print(f"  wave {waves}: ran {[e.id for e in ready]} concurrently")
    pending = [e for e in pending if e.id not in done]
print(f"\n{len(done)} tool calls in {waves} wave(s), {time.time() - t0:.2f}s")

## Phase 3 — SOLVE: one reasoning pass over everything

In [ ]:
evidence = "\n".join(f"{k} = {v}" for k, v in done.items())
r = client.chat.completions.create(model=MODEL, max_tokens=400,
    messages=[{"role": "user", "content":
               f"TASK: {TASK}\n\nEVIDENCE:\n{evidence}\n\nAnswer using only the evidence."}])
print(r.choices[0].message.content.strip())

n = len(done)
print(f"""
LLM calls: ReWOO = 2 (plan + solve), regardless of {n} tool calls.
           ReAct = {n + 1}, each re-sending the whole growing transcript.

LESSON - ReWOO's win is structural, not incremental:

  FEWER LLM CALLS   two, always. ReAct pays one reasoning call per tool, and each
                    one re-sends everything before it (module 11's curve).
  PARALLELISM       independent evidence runs in one wave. Latency is the SLOWEST
                    tool, not the SUM of tools.
  CHEAP TOKENS      tool results never pass through an LLM until SOLVE, so they
                    are not re-sent N times.

The price is rigidity, and it is worse than module 26's:

  - the plan is fixed AND the reasoning never sees an intermediate result in time
    to change course. A surprise at E1 poisons SOLVE silently.
  - it cannot do genuinely dynamic tool selection - if which tool you need depends
    on what you find, ReWOO cannot express that.
  - replanning means throwing away the whole blueprint and starting again.

Use it for high-throughput, well-understood tasks where you can enumerate the
calls: batch enrichment, report generation, fan-out lookups. Not for exploration.

Module 28 handles the case ReWOO cannot: the run FAILED, and the agent has to
work out why and try again.""")

---

**Next:** `28_reflexion_and_self_refine.ipynb`